<a href="https://colab.research.google.com/github/ajit-ai/QuantumComputing/blob/main/Deutsch_Jozsa_Algorithm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Introduction

The Deutsch-Jozsa algorithm, first introduced in [David Deutsch and Richard Jozsa (1992). "Rapid solutions of problems by quantum computation". Proceedings of the Royal Society of London A. 439: 553–558](https://royalsocietypublishing.org/doi/10.1098/rspa.1992.0167), was the first example of a quantum algorithm that performs better than the best classical algorithm. It showed that there can be advantages to using a quantum computer as a computational tool for a specific problem.

In [ ]:
!pip install qiskit
!pip install qiskit qiskit-aer
!pip install qiskit-aer-gpu
!pip install qiskit-aer-gpu-cu11
!pip install qiskit-aer
!pip install qiskit[visualization] qiskit-aer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.7/6.7 MB 57.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.4/119.4 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 57.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.5/49.5 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 MB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.0/109.0 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 98.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 26.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 74.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.6/38.6 MB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 61.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 156.9/156.9 MB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 1

In [ ]:
from qiskit import QuantumRegister, ClassicalRegister
from qiskit import QuantumCircuit, assemble

from qiskit_aer import AerSimulator
from qiskit.primitives import Sampler
import numpy as np
from qiskit_aer import Aer # Import Aer from qiskit_aer
from qiskit.visualization import plot_histogram
import matplotlib.pyplot as plt




# Oracle for a constant function (f(x) = 0 or 1)
def constant_oracle(qc, n, value):
    """
    Adds a constant oracle to the circuit.
    value: 0 or 1 (constant output of f(x))
    """
    if value == 1:
        qc.x(n)  # Flip the output qubit if f(x) = 1

# Oracle for a balanced function (e.g., f(x) = x_i for some i)
def balanced_oracle(qc, n, bit_index):
    """
    Adds a balanced oracle where f(x) = x[bit_index].
    n: number of input qubits
    bit_index: index of the bit to check (0 to n-1)
    """
    qc.cx(bit_index, n)  # CNOT: f(x) = 1 if x[bit_index] = 1

# Deutsch-Jozsa algorithm
def deutsch_jozsa(n, oracle_type="constant", oracle_param=0):
    """
    Implements Deutsch-Jozsa algorithm.
    n: number of input qubits
    oracle_type: "constant" or "balanced"
    oracle_param: 0 or 1 for constant, bit_index for balanced
    """
    # Define registers
    qreg = QuantumRegister(n + 1, 'q')  # n input qubits + 1 output qubit
    creg = ClassicalRegister(n, 'c')    # Measure only input qubits
    qc = QuantumCircuit(qreg, creg)

    # Step 1: Prepare initial state
    qc.h(qreg[0:n])       # Hadamard on input qubits
    qc.x(qreg[n])         # Flip output qubit to |1>
    qc.h(qreg[n])         # Hadamard on output qubit

    # Step 2: Apply oracle
    if oracle_type == "constant":
        constant_oracle(qc, n, oracle_param)
    elif oracle_type == "balanced":
        balanced_oracle(qc, n, oracle_param)
    else:
        raise ValueError("oracle_type must be 'constant' or 'balanced'")

    # Step 3: Apply Hadamard gates again to input qubits
    qc.h(qreg[0:n])

    # Step 4: Measure input qubits
    qc.measure(qreg[0:n], creg)

    return qc

# Run and analyze the circuit
def run_deutsch_jozsa(n, oracle_type="constant", oracle_param=0):
    qc = deutsch_jozsa(n, oracle_type, oracle_param)

    # Use Sampler primitive with AerSimulator
    simulator = AerSimulator()
    sampler = Sampler()
    job = sampler.run(qc, shots=1024)
    result = job.result()
    counts = result.quasi_dists[0].binary_probabilities()

    # Determine if function is constant or balanced
    # If all 0s (|0...0>), it's constant; otherwise, balanced
    is_constant = '0' * n in counts and len(counts) == 1
    print(f"Measurement counts: {counts}")
    print(f"Function is {'constant' if is_constant else 'balanced'}")

    return counts

# Test the algorithm
if __name__ == "__main__":
    # Test with n = 3 qubits
    n = 3

    # Test 1: Constant function (f(x) = 0)
    print("\nTesting constant function f(x) = 0:")
    run_deutsch_jozsa(n, "constant", 0)

    # Test 2: Constant function (f(x) = 1)
    print("\nTesting constant function f(x) = 1:")
    run_deutsch_jozsa(n, "constant", 1)

    # Test 3: Balanced function (f(x) = x_1)
    print("\nTesting balanced function f(x) = x_1:")
    run_deutsch_jozsa(n, "balanced", 1)


Testing constant function f(x) = 0:
Measurement counts: {'000': 1.0}
Function is constant

Testing constant function f(x) = 1:
Measurement counts: {'000': 1.0}
Function is constant

Testing balanced function f(x) = x_1:
Measurement counts: {'010': 1.0}
Function is balanced


<ipython-input-7-7156d26522f8>:72: DeprecationWarning: The class ``qiskit.primitives.sampler.Sampler`` is deprecated as of qiskit 1.2. It will be removed no earlier than 3 months after the release date. All implementations of the `BaseSamplerV1` interface have been deprecated in favor of their V2 counterparts. The V2 alternative for the `Sampler` class is `StatevectorSampler`.
  sampler = Sampler()
